# Trace each service claim to its source

Read `records` and `links` from notebook 02. Save a long-form `claims` table with
the facility branch, exact source excerpt, service, stance, source scope, citation
quality and validity. Edit `service_aliases` in `non-ml/config/project.json` to add
equivalent names for the same service.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Build the claims table

Look for explicit statements, exclusions and qualified mentions. A specialty,
single piece of equipment, broad directory category, or generic empanelment
can help identify a lead for review. The evidence field shows which text was
matched. Empty text leaves the service unknown.

In [ ]:
from facility_tool.data import extract_claims

records = STORE.read("records")
links = STORE.read("links")
claims = extract_claims(records, links, config["service_aliases"], config["as_of"])
STORE.write("claims", claims)
overview = claims.groupby(["source", "stance", "validity", "citation_quality"]).size()
show(overview.rename("claim_rows").reset_index(), limit=60)
print(f"Facilities with at least one extracted claim: {claims['facility_id'].nunique():,}")

## Inspect the excerpts and disagreements

Multiple claims per facility can repeat the same extracted text. Agreement
across independent sources requires inspecting where the text came from.
An applicable exclusion or expired listing needs review before using the claim.

In [ ]:
service = config["lookup"]["service"]
selected = claims.loc[claims["service"].eq(service)].copy()
review_columns = ["facility_id", "stance", "scope", "excerpt", "evidence_field", "validity",
                  "validity_text", "citation_quality", "source_url", "source_urls", "match_reason"]
show(selected[review_columns], limit=30)
stance_sets = selected.groupby("facility_id")["stance"].agg(set)
conflicts = stance_sets[stance_sets.map(lambda stances: {"supports", "contradicts"}.issubset(stances))]
print(f"{service} | Branches with both support and contradiction: {len(conflicts):,}")
show(selected.loc[selected["facility_id"].isin(conflicts.index), review_columns])